In [2]:
import brightway2 as bw
import pandas as pd
import logging


logging.basicConfig(level=logging.INFO)

from config import Config
from helpers import (
    get_baseline_sdf,
    generate_methane_mitigation_scenario,
    generate_potato_scenario,
    generate_sdf_supplements,
    extract_feed_supply_chain,
    generate_cow_feed_scenario,
    model_projection_activity,
    generate_historical_projection,
    get_baseline_production_exchanges,
    get_consuming_activities,
    get_activity_information,
    switch_fuel_tractors,
    prepare_future_emissions,
    import_scenario_parameters,
    add_biochar_impacts,
    generate_improved_fertilizer_rate_scenario,
    combine_scenarios,
)

In [3]:
config = Config()

bw.projects.set_current("Master thesis")
print(list(bw.databases))

['biosphere3', 'ecoinvent-3.12-cutoff', 'image - SSP1-VLLO', 'remind - SSP1-PkBudg650', 'remind - SSP1-PkBudg650 - 2025', 'remind - SSP1-PkBudg650 - 2030', 'remind - SSP1-PkBudg650 - 2035', 'remind - SSP1-PkBudg650 - 2040', 'remind - SSP1-PkBudg650 - 2045', 'remind - SSP1-PkBudg650 - 2050', 'LC - historical', 'mitigation', 'Foreground - 2050', 'test', 'Foreground - 2030']


In [4]:
ei = bw.Database("ecoinvent-3.12-cutoff")
ei_trends = bw.Database("LC - historical")
ei_background = {
    "2030": bw.Database("remind - SSP1-PkBudg650 - 2030"),
    "2035": bw.Database("remind - SSP1-PkBudg650 - 2035"),
    "2040": bw.Database("remind - SSP1-PkBudg650 - 2040"),
    "2045": bw.Database("remind - SSP1-PkBudg650 - 2045"),
    "2050": bw.Database("remind - SSP1-PkBudg650 - 2050"),
}
bio = bw.Database("biosphere3")

In [5]:
exclusions_yield = pd.read_csv("../Data/Scenario modeling/exclusions_yield.csv")
scenario_definitions = pd.read_csv("../Data/Scenario modeling/scenario_definition.csv")
cow_feeding_parameters = pd.read_csv("../Data/Scenario modeling/cow_feeding.csv")

In [33]:
reference_year = "2045"
reference_database = ei_background[reference_year]
ei_mitigation = bw.Database(f"Foreground - {reference_year}")

# Cheese

In [34]:
cow_milk_activities = [
    activity
    for activity in reference_database
    if "milk production" in activity["name"]
    and "cow" in activity["name"]
    and "cow milk" in activity["reference product"]
]
sdf_milk_baseline = get_baseline_sdf(cow_milk_activities)
sdf_milk_baseline.head()

,from activity name,from reference product,from location,from unit,from categories,from database,from key,to activity name,to reference product,to location,to categories,to database,to key,flow type,baseline
0,"milk production, from cow",cow milk,ZA,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...","milk production, from cow",cow milk,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...",production,1.000000
1,alfalfa/grass silage production,alfalfa-grass silage,ZA,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '252323942f...","milk production, from cow",cow milk,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...",technosphere,0.057441
2,"market for barley grain, feed","barley grain, feed",GLO,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'ce17d73c1f...","milk production, from cow",cow milk,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...",technosphere,0.019742
3,"market for diesel, burned in agricultural mach...","diesel, burned in agricultural machinery",GLO,megajoule,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '7f54be87cf...","milk production, from cow",cow milk,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...",technosphere,0.355124
4,"market for inorganic nitrogen fertiliser, as N","inorganic nitrogen fertiliser, as N",ZA,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '12427959ec...","milk production, from cow",cow milk,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...",technosphere,0.007361


## Ruminant methane mitigation

In [35]:
scenarios_methane_mitigation = import_scenario_parameters(
    scenario_definitions, "ruminant_methane_mitigation"
)
scenarios_methane_mitigation

,scenario,variable,unit,value
20,nop_supplementation,dose,kg,0.000071
21,nop_supplementation,variation_yield,%,0.0
22,nop_supplementation,variation_DMI,%,0.0
23,nop_supplementation,variation_CH4_intensity,%,-32.6
24,increase_feed_level,variation_yield,%,16.8
25,increase_feed_level,variation_DMI,%,58.3
26,increase_feed_level,variation_CH4_intensity,%,-16.7
27,nitrate_supplementation,dose,kg,0.0138
28,nitrate_supplementation,variation_yield,%,-0.31
29,nitrate_supplementation,variation_DMI,%,-4.19


In [36]:
sdf_milk_methane_mitigation = sdf_milk_baseline.copy()

for scenario in scenarios_methane_mitigation["scenario"].drop_duplicates():
    scenario_parameters = scenarios_methane_mitigation[
        scenarios_methane_mitigation["scenario"] == scenario
    ]

    yield_variation = (
        scenario_parameters[scenario_parameters["variable"] == "variation_yield"][
            "value"
        ].values[0]
        / 100
    )
    feeding_level_variation = (
        scenario_parameters[scenario_parameters["variable"] == "variation_DMI"][
            "value"
        ].values[0]
        / 100
    )
    CH4_intensity_variation = (
        scenario_parameters[
            scenario_parameters["variable"] == "variation_CH4_intensity"
        ]["value"].values[0]
        / 100
    )

    sdf_milk_methane_mitigation = generate_methane_mitigation_scenario(
        sdf_milk_methane_mitigation,
        config.feed_products,
        exclusions_yield,
        config.CH4_NAME,
        scenario,
        yield_variation,
        feeding_level_variation,
        CH4_intensity_variation,
    )

sdf_milk_methane_mitigation

,from activity name,from reference product,from location,from unit,from categories,from database,from key,to activity name,to reference product,to location,to categories,to database,to key,flow type,baseline,nop_supplementation,increase_feed_level,nitrate_supplementation
1,alfalfa/grass silage production,alfalfa-grass silage,ZA,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '252323942f...","milk production, from cow",cow milk,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...",technosphere,5.744135e-02,5.744135e-02,7.785074e-02,5.520570e-02
2,"market for barley grain, feed","barley grain, feed",GLO,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'ce17d73c1f...","milk production, from cow",cow milk,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...",technosphere,1.974168e-02,1.974168e-02,2.675606e-02,1.897332e-02
3,"market for diesel, burned in agricultural mach...","diesel, burned in agricultural machinery",GLO,megajoule,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '7f54be87cf...","milk production, from cow",cow milk,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...",technosphere,3.551243e-01,3.551243e-01,3.040447e-01,3.562286e-01
4,"market for inorganic nitrogen fertiliser, as N","inorganic nitrogen fertiliser, as N",ZA,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '12427959ec...","milk production, from cow",cow milk,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...",technosphere,7.360758e-03,7.360758e-03,6.302019e-03,7.383647e-03
5,"market for inorganic phosphorus fertiliser, as...","inorganic phosphorus fertiliser, as P2O5",ZA,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'fe90bab16e...","milk production, from cow",cow milk,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...",technosphere,2.453586e-03,2.453586e-03,2.100673e-03,2.461216e-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197,"Water, unspecified natural origin",None,None,cubic meter,"(natural resource, in ground)",biosphere3,"('biosphere3', '478e8437-1c21-4032-8438-872a6b...","milk production, from cow",cow milk,RoW,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '5d9f4eaf4e...",biosphere,6.497084e-02,6.497084e-02,5.562572e-02,6.517288e-02
198,"market group for electricity, low voltage","electricity, low voltage",World,kilowatt hour,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '0409dd7667...","milk production, from cow",cow milk,RoW,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '5d9f4eaf4e...",technosphere,4.574122e-02,4.574122e-02,4.574122e-02,4.574122e-02
199,market for selenium,selenium,World,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '7c010b624b...","milk production, from cow",cow milk,RoW,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '5d9f4eaf4e...",technosphere,8.317330e-08,8.317330e-08,7.121002e-08,8.343194e-08
200,"market for wood chips, dry, measured as dry mass","wood chips, dry, measured as dry mass",RER,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'e509730723...","milk production, from cow",cow milk,RoW,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '5d9f4eaf4e...",technosphere,9.677759e-03,9.677759e-03,8.285753e-03,9.707854e-03


In [37]:
cow_feeding_DM_conversion = cow_feeding_parameters[
    ["activity name", "conversion_to_kgDM"]
].dropna()

dose_3NOP = scenarios_methane_mitigation[
    (scenarios_methane_mitigation["scenario"] == "nop_supplementation")
    & (scenarios_methane_mitigation["variable"] == "dose")
]["value"].values[0]
dose_nitrate = scenarios_methane_mitigation[
    (scenarios_methane_mitigation["scenario"] == "nitrate_supplementation")
    & (scenarios_methane_mitigation["variable"] == "dose")
]["value"].values[0]

NOP_sdf = generate_sdf_supplements(
    sdf_milk_baseline,
    config.feed_products,
    cow_feeding_DM_conversion,
    ei_mitigation,
    "NOP",
    "nop_supplementation",
    dose_3NOP,
)
nitrate_sdf = generate_sdf_supplements(
    sdf_milk_baseline,
    config.feed_products,
    cow_feeding_DM_conversion,
    ei_mitigation,
    "Nitrate",
    "nitrate_supplementation",
    dose_nitrate,
)

In [38]:
sdf_milk_enteric_fermentation = pd.concat(
    [sdf_milk_methane_mitigation, NOP_sdf, nitrate_sdf], ignore_index=True
)
sdf_milk_enteric_fermentation = sdf_milk_enteric_fermentation.drop("from unit", axis=1)
scenario_cols = [
    c for c in sdf_milk_enteric_fermentation.columns if c not in config.sdf_cols
]
sdf_milk_enteric_fermentation[scenario_cols] = sdf_milk_enteric_fermentation[
    scenario_cols
].fillna(0)

sdf_milk_enteric_fermentation

,from activity name,from reference product,from location,from categories,from database,from key,to activity name,to reference product,to location,to categories,to database,to key,flow type,baseline,nop_supplementation,increase_feed_level,nitrate_supplementation
0,alfalfa/grass silage production,alfalfa-grass silage,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '252323942f...","milk production, from cow",cow milk,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...",technosphere,0.057441,0.057441,0.077851,0.055206
1,"market for barley grain, feed","barley grain, feed",GLO,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'ce17d73c1f...","milk production, from cow",cow milk,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...",technosphere,0.019742,0.019742,0.026756,0.018973
2,"market for diesel, burned in agricultural mach...","diesel, burned in agricultural machinery",GLO,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '7f54be87cf...","milk production, from cow",cow milk,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...",technosphere,0.355124,0.355124,0.304045,0.356229
3,"market for inorganic nitrogen fertiliser, as N","inorganic nitrogen fertiliser, as N",ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '12427959ec...","milk production, from cow",cow milk,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...",technosphere,0.007361,0.007361,0.006302,0.007384
4,"market for inorganic phosphorus fertiliser, as...","inorganic phosphorus fertiliser, as P2O5",ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'fe90bab16e...","milk production, from cow",cow milk,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...",technosphere,0.002454,0.002454,0.002101,0.002461
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,3-NOP,3-NOP,GLO,None,Foreground - 2045,"('Foreground - 2045', 'e9d38632632046448c487bf...","milk production, from cow",cow milk,CA-QC,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '539ec4f790...",technosphere,0.000000,0.000068,0.000000,0.000000
201,3-NOP,3-NOP,GLO,None,Foreground - 2045,"('Foreground - 2045', 'e9d38632632046448c487bf...","milk production, from cow",cow milk,RoW,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '5d9f4eaf4e...",technosphere,0.000000,0.000049,0.000000,0.000000
202,Nitrate,Nitrate,GLO,None,Foreground - 2045,"('Foreground - 2045', '2f434b69a5344e64b7103e1...","milk production, from cow",cow milk,ZA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'deb8de8b6b...",technosphere,0.000000,0.000000,0.000000,0.005942
203,Nitrate,Nitrate,GLO,None,Foreground - 2045,"('Foreground - 2045', '2f434b69a5344e64b7103e1...","milk production, from cow",cow milk,CA-QC,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '539ec4f790...",technosphere,0.000000,0.000000,0.000000,0.013402


In [39]:
sdf_milk_enteric_fermentation.to_csv(
    path_or_buf=f"../SDF/Foreground/{reference_year}/cheese_ruminant_methane_mitigation.csv",
    index=False,
)

## Feed production

In [77]:
sdf_cow_feed_baseline = extract_feed_supply_chain(
    cow_milk_activities, config.primary_crop_products
)

Tier 1
New activites to explore: 28
Tier 2
New activites to explore: 66
Tier 3
New activites to explore: 36
Tier 4
New activites to explore: 99
Tier 5
New activites to explore: 44
Tier 6
New activites to explore: 6
Tier 7
New activites to explore: 6
Tier 8
New activites to explore: 3
Tier 9
New activites to explore: 1
Tier 10
New activites to explore: 0


### Agricultural practices

In [78]:
scenarios_agricultural_practices = import_scenario_parameters(
    scenario_definitions, "agricultural_practices"
)
scenarios_agricultural_practices

,scenario,variable,unit,value
0,improve_fertilizer_placement,variation_yield,%,13.5
1,improve_fertilizer_placement,variation_N2O_emissions,%,-16.2
2,improve_fertilizer_placement,variation_CO2_emissions,%,0.0
3,improve_fertilizer_rate,variation_N_fertilizer,%,-50.0
4,improve_fertilizer_rate,variation_yield,%,-10.9
5,improve_fertilizer_rate,variation_N2O_emissions,%,-39.2
6,improve_fertilizer_rate,variation_CO2_emissions,%,0.0


In [79]:
sdf_cow_feed_agricultural_practices = sdf_cow_feed_baseline.copy()

for scenario in scenarios_agricultural_practices["scenario"].drop_duplicates():
    scenario_parameters = scenarios_agricultural_practices[
        scenarios_agricultural_practices["scenario"] == scenario
    ]

    yield_variation = (
        scenario_parameters[scenario_parameters["variable"] == "variation_yield"][
            "value"
        ].values[0]
        / 100
    )
    CO2_variation = (
        scenario_parameters[
            scenario_parameters["variable"] == "variation_CO2_emissions"
        ]["value"].values[0]
        / 100
    )
    N2O_variation = (
        scenario_parameters[
            scenario_parameters["variable"] == "variation_N2O_emissions"
        ]["value"].values[0]
        / 100
    )

    if scenario == "improve_fertilizer_rate":
        variation_N_fertilizer = (
            scenario_parameters[
                scenario_parameters["variable"] == "variation_N_fertilizer"
            ]["value"].values[0]
            / 100
        )
        sdf_cow_feed_agricultural_practices = (
            generate_improved_fertilizer_rate_scenario(
                "cheese",
                sdf_cow_feed_agricultural_practices,
                exclusions_yield,
                config.CO2_NAME,
                config.N2O_NAME,
                scenario,
                yield_variation,
                CO2_variation,
                N2O_variation,
                config.N_fertilizer_to_scale,
                variation_N_fertilizer,
            )
        )
    else:
        sdf_cow_feed_agricultural_practices = generate_cow_feed_scenario(
            sdf_cow_feed_agricultural_practices,
            config.CO2_NAME,
            config.N2O_NAME,
            scenario,
            yield_variation,
            CO2_variation,
            N2O_variation,
        )

sdf_cow_feed_agricultural_practices

,from activity name,from reference product,from location,from unit,from categories,from database,from key,to activity name,to reference product,to location,to categories,to database,to key,flow type,tier,baseline,improve_fertilizer_placement,improve_fertilizer_rate
0,Metolachlor,None,None,kilogram,"(water, ground-)",biosphere3,"('biosphere3', '00012c0a-9bff-4787-a7eb-56c3d2...","maize grain production, second crop",maize grain,BR-GO,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '05c829f1a3...",biosphere,5,3.590000e-08,3.162996e-08,4.029181e-08
1,Metolachlor,None,None,kilogram,"(water, ground-)",biosphere3,"('biosphere3', '00012c0a-9bff-4787-a7eb-56c3d2...",rape seed production,rape seed,RoW,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '4ab75ac7ec...",biosphere,5,1.224402e-10,1.078769e-10,1.374189e-10
2,Metolachlor,None,None,kilogram,"(water, ground-)",biosphere3,"('biosphere3', '00012c0a-9bff-4787-a7eb-56c3d2...",soybean production,soybean,CA-QC,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '6e7c300b5e...",biosphere,3,6.282950e-08,5.535639e-08,7.051571e-08
3,Metolachlor,None,None,kilogram,"(water, ground-)",biosphere3,"('biosphere3', '00012c0a-9bff-4787-a7eb-56c3d2...",rape seed production,rape seed,CA-QC,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '75e3e858a5...",biosphere,5,8.918000e-08,7.857269e-08,1.000898e-07
4,Metolachlor,None,None,kilogram,"(water, ground-)",biosphere3,"('biosphere3', '00012c0a-9bff-4787-a7eb-56c3d2...","maize grain production, first crop",maize grain,BR-BA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'f3bfb01f11...",biosphere,5,1.020000e-08,8.986784e-09,1.144781e-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26952,"market for tillage, rotary cultivator","tillage, rotary cultivator",GLO,hectare,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'ff630ca566...",hay production,hay,RoW,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'b82c43077a...",technosphere,3,4.931000e-05,4.344494e-05,5.534231e-05
26953,"market for tillage, rotary cultivator","tillage, rotary cultivator",GLO,hectare,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'ff630ca566...",soybean production,soybean,US-OH,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'c0573ab8b3...",technosphere,6,5.600631e-04,4.934477e-04,6.285781e-04
26954,"market for tillage, rotary cultivator","tillage, rotary cultivator",GLO,hectare,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'ff630ca566...",oat grain production,oat grain,RoW,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'c286235266...",technosphere,3,4.899000e-04,4.316300e-04,5.498317e-04
26955,"market for tillage, rotary cultivator","tillage, rotary cultivator",GLO,hectare,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'ff630ca566...",soybean production,soybean,US-ND,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd88cf1239e...",technosphere,6,9.235823e-04,8.137289e-04,1.036568e-03


In [80]:
sdf_cow_feed_agricultural_practices = sdf_cow_feed_agricultural_practices.drop(
    columns=["from unit", "tier"], axis=1
)
scenario_cols = [
    c for c in sdf_cow_feed_agricultural_practices.columns if "amount_" in c
]
sdf_cow_feed_agricultural_practices[scenario_cols] = (
    sdf_cow_feed_agricultural_practices[scenario_cols].fillna(0)
)

sdf_cow_feed_agricultural_practices.to_csv(
    path_or_buf=f"../SDF/Foreground/{reference_year}/cheese_cow_feed_agricultural_practices.csv",
    index=False,
)

### Enhanced-efficiency fertilizers

In [81]:
scenarios_eef_fertilizers = import_scenario_parameters(
    scenario_definitions, "enhanced_efficiency_fertilizers"
)
scenarios_eef_fertilizers

,scenario,variable,unit,value
11,double_inhibitors,variation_yield,%,4.9
12,double_inhibitors,variation_N2O_emissions,%,-36.0
13,double_inhibitors,variation_CO2_emissions,%,0.0
14,nitrification_inhibitors,variation_yield,%,12.9
15,nitrification_inhibitors,variation_N2O_emissions,%,-44.0
16,nitrification_inhibitors,variation_CO2_emissions,%,-8.9
17,urease_inhibitors,variation_yield,%,6.2
18,urease_inhibitors,variation_N2O_emissions,%,-24.0
19,urease_inhibitors,variation_CO2_emissions,%,0.0


In [82]:
sdf_cow_feed_eef_fertilizers = sdf_cow_feed_baseline.copy()

for scenario in scenarios_eef_fertilizers["scenario"].drop_duplicates():
    scenario_parameters = scenarios_eef_fertilizers[
        scenarios_eef_fertilizers["scenario"] == scenario
    ]

    yield_variation = (
        scenario_parameters[scenario_parameters["variable"] == "variation_yield"][
            "value"
        ].values[0]
        / 100
    )
    CO2_variation = (
        scenario_parameters[
            scenario_parameters["variable"] == "variation_CO2_emissions"
        ]["value"].values[0]
        / 100
    )
    N2O_variation = (
        scenario_parameters[
            scenario_parameters["variable"] == "variation_N2O_emissions"
        ]["value"].values[0]
        / 100
    )

    sdf_cow_feed_eef_fertilizers = generate_cow_feed_scenario(
        sdf_cow_feed_eef_fertilizers,
        config.CO2_NAME,
        config.N2O_NAME,
        scenario,
        yield_variation,
        CO2_variation,
        N2O_variation,
    )

sdf_cow_feed_eef_fertilizers

,from activity name,from reference product,from location,from unit,from categories,from database,from key,to activity name,to reference product,to location,to categories,to database,to key,flow type,tier,baseline,double_inhibitors,nitrification_inhibitors,urease_inhibitors
0,Metolachlor,None,None,kilogram,"(water, ground-)",biosphere3,"('biosphere3', '00012c0a-9bff-4787-a7eb-56c3d2...","maize grain production, second crop",maize grain,BR-GO,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '05c829f1a3...",biosphere,5,3.590000e-08,3.422307e-08,3.179805e-08,3.380414e-08
1,Metolachlor,None,None,kilogram,"(water, ground-)",biosphere3,"('biosphere3', '00012c0a-9bff-4787-a7eb-56c3d2...",rape seed production,rape seed,RoW,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '4ab75ac7ec...",biosphere,5,1.224402e-10,1.167209e-10,1.084502e-10,1.152921e-10
2,Metolachlor,None,None,kilogram,"(water, ground-)",biosphere3,"('biosphere3', '00012c0a-9bff-4787-a7eb-56c3d2...",soybean production,soybean,CA-QC,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '6e7c300b5e...",biosphere,3,6.282950e-08,5.989466e-08,5.565058e-08,5.916149e-08
3,Metolachlor,None,None,kilogram,"(water, ground-)",biosphere3,"('biosphere3', '00012c0a-9bff-4787-a7eb-56c3d2...",rape seed production,rape seed,CA-QC,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '75e3e858a5...",biosphere,5,8.918000e-08,8.501430e-08,7.899026e-08,8.397363e-08
4,Metolachlor,None,None,kilogram,"(water, ground-)",biosphere3,"('biosphere3', '00012c0a-9bff-4787-a7eb-56c3d2...","maize grain production, first crop",maize grain,BR-BA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'f3bfb01f11...",biosphere,5,1.020000e-08,9.723546e-09,9.034544e-09,9.604520e-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26952,"market for tillage, rotary cultivator","tillage, rotary cultivator",GLO,hectare,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'ff630ca566...",hay production,hay,RoW,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'b82c43077a...",technosphere,3,4.931000e-05,4.700667e-05,4.367582e-05,4.643126e-05
26953,"market for tillage, rotary cultivator","tillage, rotary cultivator",GLO,hectare,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'ff630ca566...",soybean production,soybean,US-OH,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'c0573ab8b3...",technosphere,6,5.600631e-04,5.339019e-04,4.960701e-04,5.273664e-04
26954,"market for tillage, rotary cultivator","tillage, rotary cultivator",GLO,hectare,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'ff630ca566...",oat grain production,oat grain,RoW,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'c286235266...",technosphere,3,4.899000e-04,4.670162e-04,4.339238e-04,4.612994e-04
26955,"market for tillage, rotary cultivator","tillage, rotary cultivator",GLO,hectare,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'ff630ca566...",soybean production,soybean,US-ND,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd88cf1239e...",technosphere,6,9.235823e-04,8.804407e-04,8.180534e-04,8.696632e-04


In [83]:
sdf_cow_feed_eef_fertilizers = sdf_cow_feed_eef_fertilizers.drop(
    columns=["from unit", "tier"], axis=1
)
scenario_cols = [c for c in sdf_cow_feed_eef_fertilizers.columns if "amount_" in c]
sdf_cow_feed_eef_fertilizers[scenario_cols] = sdf_cow_feed_eef_fertilizers[
    scenario_cols
].fillna(0)

sdf_cow_feed_eef_fertilizers.to_csv(
    path_or_buf=f"../SDF/Foreground/{reference_year}/cheese_cow_feed_eef_fertilizers.csv",
    index=False,
)

# Potato

In [84]:
potato_activities = [
    activity
    for activity in reference_database
    if "potato production" in activity["name"]
    and activity["reference product"] == "potato"
]

sdf_potato_baseline = get_baseline_sdf(potato_activities)
sdf_potato_baseline.head()

,from activity name,from reference product,from location,from unit,from categories,from database,from key,to activity name,to reference product,to location,to categories,to database,to key,flow type,baseline
0,potato production,potato,US-WI,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd1bb372a7c...",potato production,potato,US-WI,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd1bb372a7c...",production,1.000000
1,market for NPK (15-15-15) fertiliser,NPK (15-15-15) fertiliser,RNA,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '8c62a5a231...",potato production,potato,US-WI,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd1bb372a7c...",technosphere,0.002437
2,market for [sulfonyl]urea-compound,[sulfonyl]urea-compound,GLO,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '48c64830ac...",potato production,potato,US-WI,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd1bb372a7c...",technosphere,0.000009
3,market for [thio]carbamate-compound,[thio]carbamate-compound,GLO,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'a5e24e8f7b...",potato production,potato,US-WI,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd1bb372a7c...",technosphere,0.002073
4,"market for acetamide-anillide-compound, unspec...","acetamide-anillide-compound, unspecified",GLO,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd99f7a8fa9...",potato production,potato,US-WI,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd1bb372a7c...",technosphere,0.000007


## Agricultural practices

In [85]:
scenarios_agricultural_practices = import_scenario_parameters(
    scenario_definitions, "agricultural_practices"
)
scenarios_agricultural_practices

,scenario,variable,unit,value
0,improve_fertilizer_placement,variation_yield,%,13.5
1,improve_fertilizer_placement,variation_N2O_emissions,%,-16.2
2,improve_fertilizer_placement,variation_CO2_emissions,%,0.0
3,improve_fertilizer_rate,variation_N_fertilizer,%,-50.0
4,improve_fertilizer_rate,variation_yield,%,-10.9
5,improve_fertilizer_rate,variation_N2O_emissions,%,-39.2
6,improve_fertilizer_rate,variation_CO2_emissions,%,0.0


In [86]:
sdf_potato_agricultural_practices = sdf_potato_baseline.copy()

for scenario in scenarios_agricultural_practices["scenario"].drop_duplicates():
    scenario_parameters = scenarios_agricultural_practices[
        scenarios_agricultural_practices["scenario"] == scenario
    ]

    yield_variation = (
        scenario_parameters[scenario_parameters["variable"] == "variation_yield"][
            "value"
        ].values[0]
        / 100
    )
    CO2_variation = (
        scenario_parameters[
            scenario_parameters["variable"] == "variation_CO2_emissions"
        ]["value"].values[0]
        / 100
    )
    N2O_variation = (
        scenario_parameters[
            scenario_parameters["variable"] == "variation_N2O_emissions"
        ]["value"].values[0]
        / 100
    )

    if scenario == "improve_fertilizer_rate":
        variation_N_fertilizer = (
            scenario_parameters[
                scenario_parameters["variable"] == "variation_N_fertilizer"
            ]["value"].values[0]
            / 100
        )
        sdf_potato_agricultural_practices = generate_improved_fertilizer_rate_scenario(
            "potato",
            sdf_potato_agricultural_practices,
            exclusions_yield,
            config.CO2_NAME,
            config.N2O_NAME,
            scenario,
            yield_variation,
            CO2_variation,
            N2O_variation,
            config.N_fertilizer_to_scale,
            variation_N_fertilizer,
        )
    else:
        sdf_potato_agricultural_practices = generate_potato_scenario(
            sdf_potato_agricultural_practices,
            exclusions_yield,
            config.CO2_NAME,
            config.N2O_NAME,
            scenario,
            yield_variation,
            CO2_variation,
            N2O_variation,
        )

sdf_potato_agricultural_practices

,from activity name,from reference product,from location,from unit,from categories,from database,from key,to activity name,to reference product,to location,to categories,to database,to key,flow type,baseline,improve_fertilizer_placement,improve_fertilizer_rate
1,market for NPK (15-15-15) fertiliser,NPK (15-15-15) fertiliser,RNA,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '8c62a5a231...",potato production,potato,US-WI,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd1bb372a7c...",technosphere,0.002437,0.002147,0.001219
2,market for [sulfonyl]urea-compound,[sulfonyl]urea-compound,GLO,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '48c64830ac...",potato production,potato,US-WI,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd1bb372a7c...",technosphere,0.000009,0.000008,0.000010
3,market for [thio]carbamate-compound,[thio]carbamate-compound,GLO,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'a5e24e8f7b...",potato production,potato,US-WI,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd1bb372a7c...",technosphere,0.002073,0.001826,0.002326
4,"market for acetamide-anillide-compound, unspec...","acetamide-anillide-compound, unspecified",GLO,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd99f7a8fa9...",potato production,potato,US-WI,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd1bb372a7c...",technosphere,0.000007,0.000006,0.000008
5,"market for ammonia, anhydrous, liquid","ammonia, anhydrous, liquid",RNA,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '38e7afb6b4...",potato production,potato,US-WI,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd1bb372a7c...",technosphere,0.001717,0.001513,0.000859
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2609,"Energy, gross calorific value, in biomass",None,None,megajoule,"(natural resource, biotic)",biosphere3,"('biosphere3', '01c12fca-ad8b-4902-8b48-2d5afe...",potato production,potato,UA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '7022ca9d03...",biosphere,3.148003,2.773571,3.533112
2610,"Occupation, annual crop",None,None,square meter-year,"(natural resource, land)",biosphere3,"('biosphere3', 'c5aafa60-495c-461c-a1d4-b262a3...",potato production,potato,UA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '7022ca9d03...",biosphere,0.585519,0.515876,0.657149
2611,"Transformation, from arable land, unspecified use",None,None,square meter,"(natural resource, land)",biosphere3,"('biosphere3', '4d166779-88fd-441b-9537-f3b974...",potato production,potato,UA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '7022ca9d03...",biosphere,0.665779,0.586589,0.747227
2612,"Transformation, to arable land, unspecified use",None,None,square meter,"(natural resource, land)",biosphere3,"('biosphere3', '2f1e926a-ec96-432b-b2a6-bd5e3d...",potato production,potato,UA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '7022ca9d03...",biosphere,0.665779,0.586589,0.747227


In [87]:
sdf_potato_agricultural_practices = sdf_potato_agricultural_practices.drop(
    "from unit", axis=1
)
scenario_cols = [c for c in sdf_potato_agricultural_practices.columns if "amount_" in c]
sdf_potato_agricultural_practices[scenario_cols] = sdf_potato_agricultural_practices[
    scenario_cols
].fillna(0)

sdf_potato_agricultural_practices.to_csv(
    path_or_buf=f"../SDF/Foreground/{reference_year}/potato_agricultural_practices.csv",
    index=False,
)

## Enhanced-efficiency fertilizers

In [88]:
scenarios_eef_fertilizers = import_scenario_parameters(
    scenario_definitions, "enhanced_efficiency_fertilizers"
)

In [89]:
sdf_potato_eef_fertilizers = sdf_potato_baseline.copy()

for scenario in scenarios_eef_fertilizers["scenario"].drop_duplicates():
    scenario_parameters = scenarios_eef_fertilizers[
        scenarios_eef_fertilizers["scenario"] == scenario
    ]

    yield_variation = (
        scenario_parameters[scenario_parameters["variable"] == "variation_yield"][
            "value"
        ].values[0]
        / 100
    )
    CO2_variation = (
        scenario_parameters[
            scenario_parameters["variable"] == "variation_CO2_emissions"
        ]["value"].values[0]
        / 100
    )
    N2O_variation = (
        scenario_parameters[
            scenario_parameters["variable"] == "variation_N2O_emissions"
        ]["value"].values[0]
        / 100
    )

    sdf_potato_eef_fertilizers = generate_potato_scenario(
        sdf_potato_eef_fertilizers,
        exclusions_yield,
        config.CO2_NAME,
        config.N2O_NAME,
        scenario,
        yield_variation,
        CO2_variation,
        N2O_variation,
    )

sdf_potato_eef_fertilizers

,from activity name,from reference product,from location,from unit,from categories,from database,from key,to activity name,to reference product,to location,to categories,to database,to key,flow type,baseline,double_inhibitors,nitrification_inhibitors,urease_inhibitors
1,market for NPK (15-15-15) fertiliser,NPK (15-15-15) fertiliser,RNA,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '8c62a5a231...",potato production,potato,US-WI,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd1bb372a7c...",technosphere,0.002437,0.002323,0.002159,0.002295
2,market for [sulfonyl]urea-compound,[sulfonyl]urea-compound,GLO,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '48c64830ac...",potato production,potato,US-WI,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd1bb372a7c...",technosphere,0.000009,0.000009,0.000008,0.000009
3,market for [thio]carbamate-compound,[thio]carbamate-compound,GLO,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'a5e24e8f7b...",potato production,potato,US-WI,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd1bb372a7c...",technosphere,0.002073,0.001976,0.001836,0.001952
4,"market for acetamide-anillide-compound, unspec...","acetamide-anillide-compound, unspecified",GLO,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd99f7a8fa9...",potato production,potato,US-WI,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd1bb372a7c...",technosphere,0.000007,0.000006,0.000006,0.000006
5,"market for ammonia, anhydrous, liquid","ammonia, anhydrous, liquid",RNA,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '38e7afb6b4...",potato production,potato,US-WI,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd1bb372a7c...",technosphere,0.001717,0.001637,0.001521,0.001617
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2609,"Energy, gross calorific value, in biomass",None,None,megajoule,"(natural resource, biotic)",biosphere3,"('biosphere3', '01c12fca-ad8b-4902-8b48-2d5afe...",potato production,potato,UA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '7022ca9d03...",biosphere,3.148003,3.000956,2.788311,2.964221
2610,"Occupation, annual crop",None,None,square meter-year,"(natural resource, land)",biosphere3,"('biosphere3', 'c5aafa60-495c-461c-a1d4-b262a3...",potato production,potato,UA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '7022ca9d03...",biosphere,0.585519,0.558169,0.518618,0.551336
2611,"Transformation, from arable land, unspecified use",None,None,square meter,"(natural resource, land)",biosphere3,"('biosphere3', '4d166779-88fd-441b-9537-f3b974...",potato production,potato,UA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '7022ca9d03...",biosphere,0.665779,0.634680,0.589707,0.626910
2612,"Transformation, to arable land, unspecified use",None,None,square meter,"(natural resource, land)",biosphere3,"('biosphere3', '2f1e926a-ec96-432b-b2a6-bd5e3d...",potato production,potato,UA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '7022ca9d03...",biosphere,0.665779,0.634680,0.589707,0.626910


In [90]:
sdf_potato_eef_fertilizers = sdf_potato_eef_fertilizers.drop("from unit", axis=1)
scenario_cols = [c for c in sdf_potato_eef_fertilizers.columns if "amount_" in c]
sdf_potato_eef_fertilizers[scenario_cols] = sdf_potato_eef_fertilizers[
    scenario_cols
].fillna(0)

sdf_potato_eef_fertilizers.to_csv(
    path_or_buf=f"../SDF/Foreground/{reference_year}/potato_eef_fertilizers.csv",
    index=False,
)

## Crop amendment

In [29]:
scenarios_crop_amendment = import_scenario_parameters(
    scenario_definitions, "crop_amendment"
)
scenarios_crop_amendment

,scenario,variable,unit,value
7,biochar_amendment,dose,kg,0.488
8,biochar_amendment,variation_yield,%,25.3
9,biochar_amendment,variation_N2O_emissions,%,-19.0
10,biochar_amendment,variation_CO2_emissions,%,4.7


In [30]:
sdf_potato_crop_amendment = sdf_potato_baseline.copy()

for scenario in scenarios_crop_amendment["scenario"].drop_duplicates():
    scenario_parameters = scenarios_crop_amendment[
        scenarios_crop_amendment["scenario"] == scenario
    ]

    yield_variation = (
        scenario_parameters[scenario_parameters["variable"] == "variation_yield"][
            "value"
        ].values[0]
        / 100
    )
    CO2_variation = (
        scenario_parameters[
            scenario_parameters["variable"] == "variation_CO2_emissions"
        ]["value"].values[0]
        / 100
    )
    N2O_variation = (
        scenario_parameters[
            scenario_parameters["variable"] == "variation_N2O_emissions"
        ]["value"].values[0]
        / 100
    )

    sdf_potato_crop_amendment = generate_potato_scenario(
        sdf_potato_crop_amendment,
        exclusions_yield,
        config.CO2_NAME,
        config.N2O_NAME,
        scenario,
        yield_variation,
        CO2_variation,
        N2O_variation,
    )

sdf_potato_crop_amendment

,from activity name,from reference product,from location,from unit,from categories,from database,from key,to activity name,to reference product,to location,to categories,to database,to key,flow type,baseline,biochar_amendment
1,market for NPK (15-15-15) fertiliser,NPK (15-15-15) fertiliser,RNA,kilogram,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', 'b4959d1d82...",potato production,potato,US-ND,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '4b16250390...",technosphere,0.002372,0.001893
2,market for [sulfonyl]urea-compound,[sulfonyl]urea-compound,GLO,kilogram,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '603c9dc048...",potato production,potato,US-ND,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '4b16250390...",technosphere,0.000007,0.000006
3,market for [thio]carbamate-compound,[thio]carbamate-compound,GLO,kilogram,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '2f82889fd1...",potato production,potato,US-ND,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '4b16250390...",technosphere,0.001633,0.001303
4,"market for acetamide-anillide-compound, unspec...","acetamide-anillide-compound, unspecified",GLO,kilogram,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '3f96924232...",potato production,potato,US-ND,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '4b16250390...",technosphere,0.000009,0.000007
5,"market for ammonia, anhydrous, liquid","ammonia, anhydrous, liquid",RNA,kilogram,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '6b2416e5ec...",potato production,potato,US-ND,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '4b16250390...",technosphere,0.001671,0.001334
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2609,"Energy, gross calorific value, in biomass",None,None,megajoule,"(natural resource, biotic)",biosphere3,"('biosphere3', '01c12fca-ad8b-4902-8b48-2d5afe...",potato production,potato,UA,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '903398e4d2...",biosphere,3.148003,2.512372
2610,"Occupation, annual crop",None,None,square meter-year,"(natural resource, land)",biosphere3,"('biosphere3', 'c5aafa60-495c-461c-a1d4-b262a3...",potato production,potato,UA,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '903398e4d2...",biosphere,0.585519,0.467294
2611,"Transformation, from arable land, unspecified use",None,None,square meter,"(natural resource, land)",biosphere3,"('biosphere3', '4d166779-88fd-441b-9537-f3b974...",potato production,potato,UA,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '903398e4d2...",biosphere,0.665779,0.531348
2612,"Transformation, to arable land, unspecified use",None,None,square meter,"(natural resource, land)",biosphere3,"('biosphere3', '2f1e926a-ec96-432b-b2a6-bd5e3d...",potato production,potato,UA,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '903398e4d2...",biosphere,0.665779,0.531348


In [31]:
biochar_activity_name = (
    "biochar, closed carbonization, without surplus energy, from sludge, unspecified"
)

dose_biochar = scenarios_crop_amendment[
    (scenarios_crop_amendment["scenario"] == "biochar_amendment")
    & (scenarios_crop_amendment["variable"] == "dose")
]["value"].values[0]

biochar_activity = [
    activity for activity in ei_mitigation if biochar_activity_name in activity["name"]
]

biochar_sdf = add_biochar_impacts(
    sdf_potato_baseline, "biochar_amendment", biochar_activity, dose_biochar
)

In [32]:
sdf_potato_crop_amendment = pd.concat(
    [sdf_potato_crop_amendment, biochar_sdf], ignore_index=True
)
sdf_potato_crop_amendment = sdf_potato_crop_amendment.drop("from unit", axis=1)
scenario_cols = [c for c in sdf_potato_crop_amendment.columns if "amount_" in c]
sdf_potato_crop_amendment[scenario_cols] = sdf_potato_crop_amendment[
    scenario_cols
].fillna(0)

sdf_potato_crop_amendment

,from activity name,from reference product,from location,from categories,from database,from key,to activity name,to reference product,to location,to categories,to database,to key,flow type,baseline,biochar_amendment
0,market for NPK (15-15-15) fertiliser,NPK (15-15-15) fertiliser,RNA,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', 'b4959d1d82...",potato production,potato,US-ND,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '4b16250390...",technosphere,0.002372,0.001893
1,market for [sulfonyl]urea-compound,[sulfonyl]urea-compound,GLO,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '603c9dc048...",potato production,potato,US-ND,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '4b16250390...",technosphere,0.000007,0.000006
2,market for [thio]carbamate-compound,[thio]carbamate-compound,GLO,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '2f82889fd1...",potato production,potato,US-ND,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '4b16250390...",technosphere,0.001633,0.001303
3,"market for acetamide-anillide-compound, unspec...","acetamide-anillide-compound, unspecified",GLO,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '3f96924232...",potato production,potato,US-ND,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '4b16250390...",technosphere,0.000009,0.000007
4,"market for ammonia, anhydrous, liquid","ammonia, anhydrous, liquid",RNA,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '6b2416e5ec...",potato production,potato,US-ND,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '4b16250390...",technosphere,0.001671,0.001334
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2609,"biochar, closed carbonization, without surplus...",biochar,GLO,None,Foreground - 2050,"('Foreground - 2050', '68d432f5901e4f3ea0eb856...",potato production,potato,CA-QC,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', 'c5afa2fc39...",technosphere,NaN,0.488000
2610,"biochar, closed carbonization, without surplus...",biochar,GLO,None,Foreground - 2050,"('Foreground - 2050', '68d432f5901e4f3ea0eb856...",potato production,potato,CN,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', 'e03ab290d3...",technosphere,NaN,0.488000
2611,"biochar, closed carbonization, without surplus...",biochar,GLO,None,Foreground - 2050,"('Foreground - 2050', '68d432f5901e4f3ea0eb856...",potato production,potato,IN,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '0c3c959403...",technosphere,NaN,0.488000
2612,"biochar, closed carbonization, without surplus...",biochar,GLO,None,Foreground - 2050,"('Foreground - 2050', '68d432f5901e4f3ea0eb856...","potato production, conventional, plain region",potato,CH,None,remind - SSP1-PkBudg650 - 2050,"('remind - SSP1-PkBudg650 - 2050', '3ba70f391e...",technosphere,NaN,0.488000


In [33]:
sdf_potato_crop_amendment.to_csv(
    path_or_buf=f"../SDF/Foreground/{reference_year}/potato_crop_amendment_high_impact.csv",
    index=False,
)

# Tractor electrification

In [91]:
tractor_activities = [
    activity
    for activity in reference_database
    if "market for tractor" in activity["name"]
]
tractor_consuming_activities = get_consuming_activities(tractor_activities)
sdf_tractor_consuming_baseline = get_baseline_sdf(tractor_consuming_activities)

sdf_tractor_consuming_baseline

,from activity name,from reference product,from location,from unit,from categories,from database,from key,to activity name,to reference product,to location,to categories,to database,to key,flow type,baseline
0,"planting, by no till drill","planting, by no till drill",US,hectare,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd90e263851...","planting, by no till drill","planting, by no till drill",US,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd90e263851...",production,1.000000
1,"market for agricultural machinery, unspecified","agricultural machinery, unspecified",GLO,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'cf227c87c2...","planting, by no till drill","planting, by no till drill",US,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd90e263851...",technosphere,0.966000
2,market for diesel,diesel,USA,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '0ae1d7b571...","planting, by no till drill","planting, by no till drill",US,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd90e263851...",technosphere,5.144693
3,market for shed,shed,GLO,square meter,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '936fdbd80a...","planting, by no till drill","planting, by no till drill",US,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd90e263851...",technosphere,0.005460
4,"market for tractor, 4-wheel, agricultural","tractor, 4-wheel, agricultural",GLO,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '19d637959b...","planting, by no till drill","planting, by no till drill",US,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd90e263851...",technosphere,0.596000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5755,"market for wire drawing, copper","wire drawing, copper",GLO,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '85f9a1118f...","greenhouse construction, glass walls and roof,...","greenhouse, glass walls and roof",RoW,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'babb5b5792...",technosphere,0.000280
5756,"market for zinc coat, coils","zinc coat, coils",GLO,square meter,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '2f0459a751...","greenhouse construction, glass walls and roof,...","greenhouse, glass walls and roof",RoW,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'babb5b5792...",technosphere,0.018519
5757,"market group for electricity, low voltage","electricity, low voltage",World,kilowatt hour,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '0409dd7667...","greenhouse construction, glass walls and roof,...","greenhouse, glass walls and roof",RoW,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'babb5b5792...",technosphere,0.059720
5758,"market for steel, low-alloyed","steel, low-alloyed",World,kilogram,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '6dbb9660b0...","greenhouse construction, glass walls and roof,...","greenhouse, glass walls and roof",RoW,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'babb5b5792...",technosphere,0.521187


In [92]:
electricity_activities = [
    activity
    for activity in reference_database
    if "market group for electricity, low voltage" in activity["name"]
    and "year" not in activity["name"]
]
sdf_electricity_data = get_activity_information(electricity_activities)
sdf_electricity_data

,from activity name,from reference product,from location,from categories,from database,from key
0,"market group for electricity, low voltage","electricity, low voltage",NEU,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '25cc42adfa..."
1,"market group for electricity, low voltage","electricity, low voltage",LAM,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '89307254ee..."
2,"market group for electricity, low voltage","electricity, low voltage",USA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'ed9ae13e3a..."
3,"market group for electricity, low voltage","electricity, low voltage",OAS,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '267429cc02..."
4,"market group for electricity, low voltage","electricity, low voltage",CN,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'bef53ca1cd..."
5,"market group for electricity, low voltage","electricity, low voltage",RER,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '0cf68dbba2..."
6,"market group for electricity, low voltage","electricity, low voltage",EUR,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '002b6a8c54..."
7,"market group for electricity, low voltage","electricity, low voltage",US,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '45950edf00..."
8,"market group for electricity, low voltage","electricity, low voltage",ENTSO-E,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'a0e4d5f257..."
9,"market group for electricity, low voltage","electricity, low voltage",RLA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '86abacc6b7..."


In [93]:
sdf_electric_tractors = switch_fuel_tractors(
    sdf_tractor_consuming_baseline,
    sdf_electricity_data,
    config.diesel_combustion_flows,
    config.CALORIFIC_VALUE_DIESEL,
    config.CONVERSION_FACTOR_kWh_to_MJ,
    config.efficiency_diesel_tractor,
    config.efficiency_electric_tractor,
    config.sdf_cols,
)
sdf_electric_tractors

d:\Mes Documents\3A_DD\DTU\Master thesis\Food systems - prospective LCA\Thesis\Code\helpers.py:720: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_diesel_combustion = (sdf_tractor_consuming_baseline["flow type"] == "biosphere") & (sdf_tractor_consuming_baseline["from activity name"].str.contains('|'.join(diesel_combustion_flows), case=False, na=False))
d:\Mes Documents\3A_DD\DTU\Master thesis\Food systems - prospective LCA\Thesis\Code\helpers.py:726: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sdf_electric_tractors.loc[sdf_electric_tractors["flow type"] == "biosphere", "electric_tractor"] = 0


,from activity name,from reference product,from location,from categories,from database,from key,to activity name,to reference product,to location,to categories,to database,to key,flow type,baseline,electric_tractor
0,"market group for electricity, low voltage","electricity, low voltage",USA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'ed9ae13e3a...","planting, by no till drill","planting, by no till drill",US,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd90e263851...",technosphere,0.000000,25.185457
1,Ammonia,None,None,"(air, non-urban air or from high stacks)",biosphere3,"('biosphere3', '0f440cc0-0f74-446d-99d6-8ff0e9...","planting, by no till drill","planting, by no till drill",US,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd90e263851...",biosphere,0.000103,0.000000
2,Benzene,None,None,"(air, non-urban air or from high stacks)",biosphere3,"('biosphere3', '5e883a00-04e6-4d96-8dce-12d711...","planting, by no till drill","planting, by no till drill",US,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd90e263851...",biosphere,0.000038,0.000000
3,"Carbon dioxide, fossil",None,None,"(air, non-urban air or from high stacks)",biosphere3,"('biosphere3', 'aa7cac3a-3625-41d4-bc54-33e2cf...","planting, by no till drill","planting, by no till drill",US,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd90e263851...",biosphere,6.141134,0.000000
4,"Carbon monoxide, fossil",None,None,"(air, non-urban air or from high stacks)",biosphere3,"('biosphere3', '099b36ab-4c03-4587-87f4-2f81e3...","planting, by no till drill","planting, by no till drill",US,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'd90e263851...",biosphere,0.019259,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2182,market for diesel,diesel,USA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '0ae1d7b571...","planting with starter fertiliser or amendment,...","planting with starter fertiliser or amendment,...",US,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '69426e4171...",technosphere,3.557189,0.000000
2183,market for diesel,diesel,World,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '547c635863...","tillage, rolling","tillage, rolling",RoW,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'c57329d2f7...",technosphere,3.180000,0.000000
2184,"market for diesel, burned in building machine","diesel, burned in building machine",GLO,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '5a7d375bd6...","greenhouse construction, plastic walls and roo...","greenhouse, plastic walls and roof",FR,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', 'bdd408dd57...",technosphere,0.373733,0.000000
2185,market for diesel,diesel,USA,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '0ae1d7b571...","application of plant protection product, by fi...","application of plant protection product, by fi...",US,None,remind - SSP1-PkBudg650 - 2045,"('remind - SSP1-PkBudg650 - 2045', '49af632485...",technosphere,0.636683,0.000000


In [94]:
sdf_electric_tractors.to_csv(
    path_or_buf=f"../SDF/Foreground/{reference_year}/electric_tractors.csv", index=False
)

### Combined scenarios

In [96]:
mitigation_cow_feed = "cheese_cow_feed_eef_fertilizers"

cow_feed_sdf = pd.read_csv(
    f"../SDF/Foreground/{reference_year}/{mitigation_cow_feed}.csv"
)
electric_tractors_sdf = pd.read_csv(
    f"../SDF/Foreground/{reference_year}/electric_tractors.csv"
)

combined_sdf = combine_scenarios(cow_feed_sdf, electric_tractors_sdf, config.sdf_cols)

combined_sdf.to_csv(
    path_or_buf=f"../SDF/Foreground/{reference_year}/{mitigation_cow_feed}+electric_tractors.csv",
    index=False,
)

## Historical trends - Milk

In [ ]:
predicted_impacts_milk = pd.read_csv("../Data/future_impacts_milk.csv")
baseline_impacts_milk = pd.read_csv(
    "../Results/EF v3.1/Baseline/Baseline - milk production.csv"
)

In [ ]:
future_columns = predicted_impacts_milk["year"].astype(str).to_list()

future_milk_emissions = prepare_future_emissions(
    predicted_impacts=predicted_impacts_milk, baseline_impacts=baseline_impacts_milk
)
future_milk_emissions = future_milk_emissions.pivot(
    index="to location",
    columns="year",
    values="emissions_intensity",
)

In [ ]:
milk_prediction_activities = [
    a for a in ei_trends if a.get("reference product") == "cow milk"
]
milk_prediction_sdf = model_projection_activity(
    milk_prediction_activities, config.CO2_NAME
)
milk_prediction_sdf = milk_prediction_sdf.merge(
    future_milk_emissions, left_on="to location", right_index=True, how="left"
).drop(columns=["from unit", "amount_baseline"], errors="ignore")

milk_prediction_sdf

In [ ]:
sdf_historical_projection_milk = pd.DataFrame()

for year in ei_background.keys():
    background_database = ei_background[year]

    logging.info(f"Generating historical projection for {year}")

    cow_milk_activities = [
        activity
        for activity in background_database
        if "milk production" in activity["name"]
        and "cow milk" in activity["reference product"]
    ]
    sdf_baseline_milk_exchanges = get_baseline_production_exchanges(
        cow_milk_activities, background_database, future_columns
    )
    sdf_historical_projection_milk_year = generate_historical_projection(
        sdf_baseline_milk_exchanges, milk_prediction_sdf, year, future_columns
    )

    sdf_historical_projection_milk = pd.concat(
        [sdf_historical_projection_milk, sdf_historical_projection_milk_year]
    )

    logging.info(f"Historical projection for {year} generated")

sdf_historical_projection_milk = pd.concat(
    [sdf_historical_projection_milk, milk_prediction_sdf]
).reset_index(drop=True)

sdf_historical_projection_milk.to_csv(
    path_or_buf="../SDF/LC - historical/historical_projection_milk_w_premise.csv",
    index=False,
)

## Historical trends - Potato

In [ ]:
predicted_impacts_potato = pd.read_csv("../Data/future_impacts_potato.csv")
baseline_impacts_potato = pd.read_csv(
    "../Results/EF v3.1/Baseline/Baseline - potato production.csv"
)

future_columns = predicted_impacts_potato["year"].astype(str).to_list()

In [ ]:
future_potato_emissions = prepare_future_emissions(
    predicted_impacts=predicted_impacts_potato, baseline_impacts=baseline_impacts_potato
)

future_potato_emissions = future_potato_emissions.pivot(
    index=["name", "to location"], columns="year", values="emissions_intensity"
)

future_potato_emissions

In [ ]:
potato_prediction_activities = [
    a for a in ei_trends if "potato production" in a["name"]
]

potato_prediction_sdf = model_projection_activity(
    potato_prediction_activities, config.CO2_NAME
)

potato_prediction_sdf = potato_prediction_sdf.merge(
    future_potato_emissions,
    left_on=["to activity name", "to location"],
    right_on=["name", "to location"],
    how="left",
).drop(columns=["from unit", "amount_baseline"], errors="ignore")
potato_prediction_sdf

In [ ]:
sdf_historical_projection_potato = pd.DataFrame()

for year in ei_background.keys():
    background_database = ei_background[year]

    logging.info(f"Generating historical projection for {year}")

    potato_activities = [
        activity
        for activity in background_database
        if "potato production" in activity["name"]
    ]
    sdf_baseline_potato_exchanges = get_baseline_production_exchanges(
        potato_activities, background_database, future_columns
    )

    potato_market_activity = [
        activity
        for activity in ei_trends
        if "market for potato" in activity["name"] and year in activity["name"]
    ][0]

    potato_market_db = potato_market_activity.get("database")
    potato_market_key = str((potato_market_db, potato_market_activity.get("code")))

    mask = (
        sdf_baseline_potato_exchanges["to activity name"] == "market for potato"
    ) & (sdf_baseline_potato_exchanges["to location"] == "RoW")

    sdf_baseline_potato_exchanges.loc[mask, "to database"] = potato_market_db
    sdf_baseline_potato_exchanges.loc[mask, "to key"] = potato_market_key

    sdf_historical_projection_potato_year = generate_historical_projection(
        sdf_baseline_potato_exchanges, potato_prediction_sdf, year, future_columns
    )

    sdf_historical_projection_potato = pd.concat(
        [sdf_historical_projection_potato, sdf_historical_projection_potato_year]
    )

    logging.info(f"Historical projection for {year} generated")

sdf_historical_projection_potato = pd.concat(
    [sdf_historical_projection_potato, potato_prediction_sdf]
).reset_index(drop=True)

sdf_historical_projection_potato

In [ ]:
sdf_historical_projection_potato.to_csv(
    path_or_buf="../SDF/LC - historical/historical_projection_potato_w_premise.csv",
    index=False,
)